<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/capacity_and_bottleneck_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capacity and Bottleneck Analysis

Demonstrate how a process can be screened for bottlenecks using utilization snapshots and simple equipment limits.


## Setup

Install imports and create a gas stream.


In [1]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
from neqsim import jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.3 MB/s eta 0:00:00


## Build a Compressor Case

Set a design power so the compressor has a meaningful utilization basis.


In [18]:
Compressor = jneqsim.process.equipment.compressor.Compressor
PipeBP = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills
fluid = make_gas(25.0, 45.0)
feed = Stream("Feed gas", fluid)
feed.setFlowRate(12000.0, "kg/hr")
compressor = Compressor("Export compressor", feed)
compressor.setOutletPressure(100.0)

pipe1 = PipeBP('pipe1', compressor.getOutStream())
pipe1.setLength(100)
pipe1.setDiameter(0.15)

process = ProcessSystem()
process.add(feed)
process.add(compressor)
process.add(pipe1)
process.run()

process.initAllMechanicalDesigns()
compressor.getMechanicalDesign().setMaxDesignPower(2500.0)

## Read the Utilization Snapshot

The snapshot summarizes unit constraints and identifies the current bottleneck.


In [19]:
auto = process.getAutomation()
snapshot = json.loads(str(auto.getUtilizationSnapshot()))
print(json.dumps(snapshot.get("bottleneck"), indent=2))
pd.DataFrame(snapshot.get("units", []))


{
  "name": "Export compressor",
  "utilization": 0.13853257034159122,
  "utilizationPercent": 13.853257034159123,
  "limitingConstraint": "power"
}


,name,type,capacityAnalysisEnabled,maxUtilization,maxUtilizationPercent,limitingConstraint,feasible,hardLimitExceeded,constraints,power_kW
0,Feed gas,Stream,True,0.000000,0.000000,None,True,False,[],NaN
1,Export compressor,Compressor,True,0.138533,13.853257,power,True,False,"[{'name': 'speed', 'utilization': 0.1, 'utiliz...",346.331426
2,pipe1,PipeBeggsAndBrills,True,0.135811,13.581116,velocity,True,False,"[{'name': 'velocity', 'utilization': 0.1358111...",NaN


## Screen Operating Rate

Increase feed rate and track compressor utilization.


In [21]:
rows = []
for flow in [6000, 9000, 12000, 15000, 18000,25000, 48000]:
    feed.setFlowRate(float(flow), "kg/hr")
    process.run()
    snap = json.loads(str(process.getUtilizationSnapshotJson()))
    bottleneck = snap.get("bottleneck") or {}
    rows.append({"flow_kg_per_hr": flow, "bottleneck": bottleneck.get("name"), "utilization_pct": bottleneck.get("utilizationPercent")})
pd.DataFrame(rows)


,flow_kg_per_hr,bottleneck,utilization_pct
0,6000,Export compressor,6.926629
1,9000,Export compressor,10.389943
2,12000,Export compressor,13.853257
3,15000,Export compressor,17.316571
4,18000,Export compressor,20.779886
5,25000,Export compressor,28.860952
6,48000,Export compressor,55.413028
